## 📁 Configuration du chemin du projet

Cette ligne permet de définir le dossier principal du projet afin que Python puisse accéder correctement aux fichiers (`data`, notebooks, modèles, etc.).

⚠️ Important :  
Chaque utilisateur doit modifier ce chemin selon l’emplacement de son propre projet sur son ordinateur.

Exemple :


In [ ]:
import os 

# Changer ce chemin par le chemin absolu de votre projet
os.chdir(r"D:\pfe-mlops-scolaire")

In [29]:
#Importation des bibliotheque  
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient


In [3]:
# Reproductibilite
Random_State = 42

In [4]:
# Chargement des donnees  
X_train = pd.read_csv("data/X_train_final.csv")
X_test = pd.read_csv("data/X_test_final.csv")
y_train = pd.read_csv("data/y_train_binary.csv").values.ravel()
y_test = pd.read_csv("data/y_test_binary_1.csv").values.ravel()

print("Data loaded successfully")


Data loaded successfully


In [ ]:
# Fonction pour evaluer les performances du model
def func_evaluate(model, X_test, y_test):
     
     y_pred = model.predict(X_test)
     
     # Calcul de l'Accuracy / Mesure le pourcentage de bonnes prediction 
     acc = accuracy_score(y_test, y_pred)
     
     # Calcul du F1.score / Mesure l'equilibre entre precision et rappel
     f1 = f1_score(y_test, y_pred)
     
     #Creation de matrice de confusion / Permet de visualisation les bonnes et mauvaises
     mat_con = confusion_matrix(y_test, y_pred)
     
     return acc, f1, mat_con

In [33]:
# ================================================
# 📌 Entrainement et evaluation du modele
# Logistic Regression
# ================================================

print("\n*********Logistic regression ***********")

# class_weight='balanced' : gere le desequilibre des classes
model_LREG = LogisticRegression(max_iter=1000,class_weight='balanced', random_state=Random_State)
model_LREG.fit(X_train, y_train)

acc_LREG, f1_LREG, mat_conf_LREG = func_evaluate(model_LREG, X_test, y_test)

print("Accuracy : ", acc_LREG)
print("F1 Score :", f1_LREG)
print("Confusion Matrix : \n ", mat_conf_LREG)

2026/05/13 01:09:22 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '285a7705bd7541d89d330b069ed34862', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/13 01:09:22 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\pc HD\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Inte


*********Logistic regression ***********


2026/05/13 01:09:22 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\pc HD\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/13 01:09:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising c

Accuracy :  0.759493670886076
F1 Score : 0.8155339805825242
Confusion Matrix : 
  [[18  9]
 [10 42]]


In [34]:
# ================================================
# 📌 Entrainement et evaluation du modele
# Logistic Regression
# ================================================

print("\n *************** Arbre de décision ****************")

model_Arbre = DecisionTreeClassifier(max_depth=5, random_state=Random_State)
model_Arbre.fit(X_train, y_train)

acc_DARB, f1_DARB, mat_conf_DARB = func_evaluate(model_Arbre, X_test, y_test)

print("Accuracy : ", acc_DARB)
print("F1 Score :", f1_DARB)
print("Confusion Matrix : \n ", mat_conf_DARB)



 *************** Arbre de décision ****************


2026/05/13 01:10:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/13 01:10:22 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\pc HD\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing va

Accuracy :  0.7088607594936709
F1 Score : 0.7927927927927928
Confusion Matrix : 
  [[12 15]
 [ 8 44]]


In [26]:

#Activation de l'autoLoggin pour capturer automatiquement les parametres et les metriques
mlflow.sklearn.autolog()

# Definition de l'experience dans MLflow UI
mlflow.set_experiment("Prédiction de la réussite des étudiants Version Final De tracking")

#< Boucle pour tester plusieurs valeurs de max_iter
for depth in [3, 4, 6, 7]:
    
    # Démarrage d'un run Mlflow (une expérience)
    with mlflow.start_run(run_name=f"DecisionTree_depth_{depth}"):
        
         # Ajout d'un tag pour identifier le modéle
         mlflow.set_tag("registered_model_name", "Decision_tree_Production_Model_VF_DT")
         mlflow.set_tag("model", "Decision Tree")
        
         # -----Entrainement-------
         model = DecisionTreeClassifier(max_depth=depth, random_state=Random_State)
         model.fit(X_train, y_train)
        
         # Logging des métriques dans Mlflow
         acc, f1, mat_conf = func_evaluate(model, X_test, y_test)
         mlflow.log_metric("accuracy", acc)
         mlflow.log_metric("f1_score", f1)
         
         # Enregistrement du Modèle dans 'Model Registry' pour faciliter les deploiment futur
         mlflow.sklearn.log_model(
         sk_model=model,
         name="Decision_Tree", #(nouvelle version Mlflow)
         registered_model_name="Decision_tree_Production_Model_VF_DT"
         )


2026/05/12 18:44:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\pc HD\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/12 18:44:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\pc HD\AppData\Lo

In [27]:
#Activation de l'autoLoggin pour capturer automatiquement les parametres et les metriques
mlflow.sklearn.autolog()
# Definition de l'experience dans MLflow UI
mlflow.set_experiment("Prédiction de la réussite des étudiants Version Final De tracking")

#< Boucle pour tester plusieurs valeurs de max_iter
for max_iter in [100, 500, 1000]:
    
    # Démarrage d'un run Mlflow (une expérience)
    with mlflow.start_run(run_name=f"Logistic_Regression_Max_iter_{max_iter}"):
        
         # Ajout d'un tag pour identifier le modéle
         mlflow.set_tag("registered_model_name", "LREG_Production_Model_VF_REG")
         mlflow.set_tag("model", "Logistic Regression")
        
         # -----Entrainement-------
         model = LogisticRegression(max_iter=max_iter,class_weight='balanced', random_state=Random_State)
         model.fit(X_train, y_train)
        
         # Logging des métriques dans Mlflow
         acc, f1, mat_conf = func_evaluate(model, X_test, y_test)
         mlflow.log_metric("accuracy", acc)
         mlflow.log_metric("f1_score", f1)
         
         # Enregistrement du Modèle dans 'Model Registry' pour faciliter les deploiment futur
         mlflow.sklearn.log_model(
         sk_model=model,
         name="Logistic_model", #(nouvelle version Mlflow)
         registered_model_name="LREG_Production_Model_VF_REG"
         )


2026/05/12 18:45:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\pc HD\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\pc HD\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarni

In [ ]:

# Fonction pour choisir le automatiqument le meilleur model et le mettre en Production dans MLflow Registry
def promote_the_best_model_to_Production(experiment_name):
    
    # Connexion au client mlflow
    client = MlflowClient()
    
    # Recuperation de l'experience par son nom
    experiment = client.get_experiment_by_name(experiment_name)
    
    # Recherche du meilleur run selon l'accuracy
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.accuracy DESC"],
        max_results= 1
    )
    
    # Verification si aucun run n'existe
    if not runs:
        print("There is no experiment Recode")
        return
    
    # Recuperation du meilleur run
    best_run = runs[0]
    
    # Recuperation du run ID
    best_run_id = best_run.info.run_id
    
    # Recuperation du meilleur accuracy
    best_accuracy= best_run.data.metrics['accuracy']
    
    print("The best run id :", best_run_id)
    print("The best Accuracy:", best_accuracy)
    
    # Recuperation automatique du nom du model
    model_name = best_run.data.tags["registered_model_name"]
    
    print("Best Model : ", model_name)
    
    # Recherche des versions du modele dans Model Registry
    version =client.search_model_versions(f"name ='{model_name}'")
    
    best_version = None
    # Recherche de la version correspondant au meilleur Run
    for v in version:
        if v.run_id == best_run_id:
            best_version = v.version
            break
    # Verification si aucune version n'est trouvee    
    if best_version is None:
        print("No model version found for the best run")       
     
    # Passage du meilleur model en Production   
    client.transition_model_version_stage(
        name = model_name,
        version= best_version,
        stage = "Production",
        archive_existing_versions=True
    )
    
    # Affichage du resultat final
    print(
        f" Success! {model_name}"
        f" Version {best_version}"
        f"is now in Production with Accuracy : {best_accuracy:.4f}"
        )
# Appel de la fonction    
promote_the_best_model_to_Production(
        "Prédiction de la réussite des étudiants Version Final De tracking"
    )

The best run id : 0feb9e725ebd435090157468f8ed804e
The best Accuracy: 0.759493670886076
Best Model :  LREG_Production_Model_VF_REG
 Success! LREG_Production_Model_VF_REG Version 6is now in Production with Accuracy : 0.7595


C:\Users\pc HD\AppData\Local\Temp\ipykernel_7196\274646713.py:38: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
